## GearNet+ChemBERTa - SVR, NRKF

In [8]:
import pandas as pd
import numpy as np
import random
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import mean_absolute_error, r2_score
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, TensorDataset
from transformers import AutoTokenizer, AutoModel
import torch.optim as optim
from tqdm import tqdm
from sklearn.svm import SVR
import json

In [2]:
# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## Load the data:

In [3]:
# Load the GearNet protein embeddings (structural-based)
protein_embs = torch.load("gearnet_embeddings.pt")
protein_embs = dict(sorted(protein_embs.items()))

In [4]:
# Generate the ligand embeddings
data = pd.read_csv('/dcs/22/u2243582/cs310/feature_extraction/refined-set-csv.csv')
codes = protein_embs.keys()
data = data[data['PDB_Code'].isin(codes)]
data = data.sort_values(by='PDB_Code')
ligand_SMILES = data['Ligand_SMILES'].tolist()
chemberta_model_name = "DeepChem/ChemBERTa-10M-MTR" # Load ligand embedding model (ChemBERTa)
chemberta_tokenizer = AutoTokenizer.from_pretrained(chemberta_model_name)
chemberta_model = AutoModel.from_pretrained(chemberta_model_name).to(device)

# Batch processing for ligand SMILES strings
def generate_ligand_embeddings(smiles_list, batch_size=4):
    chemberta_model.eval()
    all_embeddings = []
    
    with torch.no_grad():
        for i in tqdm(range(0, len(smiles_list), batch_size), desc="Processing Ligand Batches"):
            batch = smiles_list[i:i + batch_size]
            inputs = chemberta_tokenizer(batch, return_tensors="pt", padding=True, truncation=True).to(device)
            outputs = chemberta_model(**inputs)
            # Extract the last layer and apply mean pooling
            embeddings = outputs.last_hidden_state.mean(dim=1)
            all_embeddings.append(embeddings)
            # Clear GPU memory after processing the batch
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
    
    return torch.cat(all_embeddings, dim=0)

ligand_embeddings = generate_ligand_embeddings(ligand_SMILES)
print(f"Ligand embeddings shape: {ligand_embeddings.shape}") # (5057, 384)

2025-04-01 14:24:27.330103: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-04-01 14:24:27.780890: I tensorflow/core/util/port.cc:104] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-04-01 14:24:29.580774: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer.so.7'; dlerror: libnvinfer.so.7: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /local/java/cuda-11.6.0/lib64/:/local/java/cudnn-linux-x86_64-8.5.0.96_cuda11-archive/l

Ligand embeddings shape: torch.Size([5057, 384])


In [5]:
# Single array of embeddings data
protein_embeddings = torch.stack(list(protein_embs.values()))
protein_embeddings = protein_embeddings.squeeze(1) # (5057, 1536)

embeddings = torch.cat((protein_embeddings, ligand_embeddings), dim=1)
print(embeddings.shape) # (5057, 1920)

torch.Size([5057, 1920])


In [6]:
X = embeddings
X = X.cpu().numpy()
X = pd.DataFrame(X, index=data.index)
data_final = X.join(data['PDB_Code'])
data_final = data_final.join(data['Log_binding'])
display(data_final)

,0,1,2,3,4,5,6,7,8,9,...,1912,1913,1914,1915,1916,1917,1918,1919,PDB_Code,Log_binding
1233,-0.000070,-0.000130,-0.000086,-0.000067,-8.618832e-05,-0.000088,-0.000115,-1.193285e-04,-0.000138,-0.000090,...,-0.186536,-0.049798,0.196158,0.135863,-0.108703,-0.025921,-0.393508,-0.102847,10gs,6.40
5194,0.000006,-0.000002,-0.000006,-0.000014,-1.439452e-05,-0.000006,0.000006,-8.106232e-06,-0.000025,-0.000013,...,-0.338283,0.003619,-0.224572,-0.190923,-0.175987,-0.180435,0.030899,-0.058560,184l,4.72
248,-0.000012,-0.000019,-0.000014,-0.000009,1.788139e-07,-0.000006,-0.000027,-8.046627e-07,-0.000026,-0.000013,...,-0.321423,0.231955,0.086743,-0.375273,0.028141,0.065830,-0.285709,-0.386267,185l,3.54
598,-0.000010,-0.000013,-0.000013,-0.000005,7.987022e-06,-0.000011,0.000001,-5.960464e-07,-0.000021,0.000017,...,-0.352505,-0.022154,-0.129489,-0.117797,-0.245010,-0.050578,0.190613,-0.078249,186l,4.85
4905,-0.000003,-0.000005,-0.000024,-0.000004,-1.041219e-05,-0.000002,-0.000004,-1.406670e-05,-0.000020,-0.000016,...,-0.358238,-0.194482,-0.120265,-0.238693,-0.254972,-0.042122,0.098773,-0.237727,187l,3.37
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2357,0.000071,0.000098,0.000085,0.000023,5.209446e-05,0.000078,0.000116,1.436472e-04,0.000064,0.000104,...,-0.203929,-0.159792,-0.409724,-0.257036,-0.271170,-0.099442,-0.171983,0.092119,7std,10.72
3025,0.000007,-0.000032,-0.000007,0.000007,2.032518e-05,-0.000004,0.000001,3.993511e-06,0.000007,0.000013,...,-0.114934,-0.084237,-0.469033,-0.236242,0.243601,-0.124348,0.329557,0.544560,7upj,8.49
3996,-0.000010,-0.000152,-0.000037,-0.000026,-4.045665e-05,-0.000002,-0.000015,-2.247095e-05,-0.000041,-0.000015,...,-0.164490,0.173755,0.111547,0.158992,0.217736,0.145620,0.319391,0.400315,8a3h,4.06
1831,-0.000006,-0.000024,0.000015,-0.000025,6.198883e-06,-0.000053,0.000005,5.960464e-07,-0.000016,-0.000020,...,-0.188235,0.106919,-0.018332,0.148484,0.018087,0.313451,-0.042479,-0.086230,8cpa,9.15


## Making the clusters

### DON'T RERUN - Making a diff .fasta file bc 2 complexes failed to make residue graph, 5057

In [12]:
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio import SeqIO

# Function to write to FASTA file
def write_fasta_from_df(df, fasta_file):
    """
    Write a FASTA file from a DataFrame.
    
    Parameters:
    df : pandas.DataFrame
        DataFrame containing index, pdb_code, and fasta_string columns.
    fasta_file : str
        Output FASTA file name.
    """
    records = [
        SeqRecord(Seq(sequence), id=str(ind), description="")
        for ind, sequence in zip(df['PDB_Code'], df['Protein_FASTA'])
    ]
    
    # Write all records to a FASTA file
    SeqIO.write(records, fasta_file, "fasta")
    print(f"FASTA file written to {fasta_file}")
    
write_fasta_from_df(data, "gearnetpdb.fasta")

FASTA file written to gearnetpdb.fasta


### Use the gearnetpdb.fasta file, run CDHIT() in ssh terminal to generate the clusters, load .json into here:

In [42]:
with open("/dcs/22/u2243582/cs310/nrkf_physicochem_feat/clustered_proteins.json", "r") as json_file:
    cluster_dict = json.load(json_file)

print(len(cluster_dict)) # length of dict should match the output of num of clusters from clustering program

cluster_assignment_dict = {}
for cluster_ind in cluster_dict:
    for pdb in cluster_dict[cluster_ind]:
        cluster_assignment_dict[pdb] = cluster_ind
        
identifiers = list(cluster_assignment_dict.keys())

1190


### Run NRKFold() to generate 5 folds from the protein clusters (code from BioTools):

In [10]:
def NRKFold(E,pc,K = 5, shuffle=True):
    e = [pc[str(x)] for x in E] #cluster indices of all proteins in the examples
    c2idx={} #indices of examples of each cluster in e
    for i,x in enumerate(e):
        try: 
            c2idx[x].append(i)
        except:
            c2idx[x]=[i]    
    ce = dict([(c,len(c2idx[c])) for c in c2idx]) #counts of examples of different clusters    
    cF = [0]*K; #counts of examples in each fold
    CF = [[] for _ in range(K)]; #clusters in each fold
    F = [[] for _ in range(K)];#indices of examples in each fold
    keys = list(ce.keys())
    if shuffle:
        random.shuffle(keys)
    for k in keys:
        v = ce[k]
        idx = np.argmin(cF)
        cF[idx]+=v
        CF[idx].append(k) #add cluster to fold
        F[idx].extend(c2idx[k])
    return F

In [43]:
folds = NRKFold(identifiers, cluster_assignment_dict)

# Get the pdb codes and indices in the dataframe per fold
final_folds = []
for fold in folds:
    temp_fold = []
    for obj in fold:
        pdb = identifiers[obj]
        data_ind = data.index[data['PDB_Code'] == pdb].item()
        temp_fold.append(data_ind)
    final_folds.append(temp_fold)
    
# Make the final folds of indices relating to the dataframe
# The final NRKfolds:
splits = [
    {
        "train_ix": final_folds[0] + final_folds[1] + final_folds[2] + final_folds[3],
        "test_ix": final_folds[4]
    },
    {
        "train_ix": final_folds[0] + final_folds[1] + final_folds[2] + final_folds[4],
        "test_ix": final_folds[3]
    },
    {
       "train_ix": final_folds[0] + final_folds[1] + final_folds[4] + final_folds[3],
        "test_ix": final_folds[2]
    },
    {
        "train_ix": final_folds[0] + final_folds[4] + final_folds[2] + final_folds[3],
        "test_ix": final_folds[1]
    },
    {
        "train_ix": final_folds[4] + final_folds[1] + final_folds[2] + final_folds[3],
        "test_ix": final_folds[0]
    }
]

## Run the model - SVR on the embeddings:

In [44]:
# Use modal hyperparameters from baseline nested cv:
# 'kernel': 'rbf', 'gamma': 'scale', 'epsilon': 0.30000000000000004, 'C': 4

# Store metrics across folds
pearsonCoeffs = []
pearsonPValues = []
spearmanCoeffs = []
spearmanPValues = []
maeScores = []
varianceScores = []
r2Scores = []

for fold in splits:
    
    train_ix = np.array(fold["train_ix"])
    test_ix = np.array(fold["test_ix"])
    
    X_train = data_final.loc[train_ix, 0:1919]
    X_test = data_final.loc[test_ix, 0:1919]
    y_train = data_final.loc[train_ix, 'Log_binding']
    y_test = data_final.loc[test_ix, 'Log_binding']
    
    svRegressor = SVR(kernel='rbf', gamma='scale', epsilon=0.30000000000000004, C=4)
    svRegressor.fit(X_train, y_train)
    svRegressorPred = svRegressor.predict(X_test)
    
    # Pearson Correlation
    pearsonCoef, pearsonP = pearsonr(y_test, svRegressorPred)
    pearsonCoeffs.append(pearsonCoef)
    pearsonPValues.append(pearsonP)
    
    # Spearman Correlation
    spearmanCoef, spearmanP = spearmanr(y_test, svRegressorPred)
    spearmanCoeffs.append(spearmanCoef)
    spearmanPValues.append(spearmanP)
    
    # Mean Absolute Error
    mae = mean_absolute_error(y_test, svRegressorPred)
    maeScores.append(mae)
    
    # Variance of Errors
    variance = np.var(y_test - svRegressorPred)
    varianceScores.append(variance)

    # R2 (Coefficient of Determination)
    r2 = r2_score(y_test, svRegressorPred)
    r2Scores.append(r2)
    
# Calculate mean and standard deviation for each metric
metricsSummary = {
    "Pearson Correlation": (np.mean(pearsonCoeffs), np.std(pearsonCoeffs)),
    "Pearson P-value": (np.mean(pearsonPValues), np.std(pearsonPValues)),
    "Spearman Correlation": (np.mean(spearmanCoeffs), np.std(spearmanCoeffs)),
    "Spearman P-value": (np.mean(spearmanPValues), np.std(spearmanPValues)),
    "Mean Absolute Error": (np.mean(maeScores), np.std(maeScores)),
    "Variance of Errors": (np.mean(varianceScores), np.std(varianceScores)),
    "R2": (np.mean(r2Scores), np.std(r2Scores))
}

# Print metrics summary
for metric, (mean, std) in metricsSummary.items():
    print(f"{metric}: Mean = {mean}, Std = {std}")

Pearson Correlation: Mean = 0.554505700209005, Std = 0.05704223800490517
Pearson P-value: Mean = 7.765104664727281e-61, Std = 1.5530209317271463e-60
Spearman Correlation: Mean = 0.5557331122916266, Std = 0.06018017676507122
Spearman P-value: Mean = 7.720840573164462e-60, Std = 1.5441681146320832e-59
Mean Absolute Error: Mean = 1.3018624912721637, Std = 0.07815805080455304
Variance of Errors: Mean = 2.5500038100903666, Std = 0.21348968736092785
R2: Mean = 0.27198615613866556, Std = 0.06828804549564722
